# SimSiam — Exploring Simple Siamese Representation Learning (2020)

_Papers · Self-supervised & Representation_

**Self-supervised learning with no negatives, no momentum encoder, no big batches — a stop-gradient alone stops collapse.**

---

This notebook is a practice scaffold for the **SimSiam — Exploring Simple Siamese Representation Learning (2020)** lesson. We build it up one small step at a time: read the note above each cell, run the cell, watch what changes, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We rebuild SimSiam from the ground up in four steps: (1) sanity-check the negative-cosine loss on a worked example, (2) define the shared encoder and the asymmetric predictor, (3) write the symmetric SimSiam loss with its all-important stop-gradient, then (4) train it twice — with and without the stop-gradient — and watch one run stay healthy while the other collapses.

### Step 1 — Sanity-check the loss on a worked example

The SimSiam loss is built from the **negative cosine similarity** `D(p, z) = -cos(p, z)`. Its lowest possible value is `-1`, reached only when the two vectors point in exactly the same direction. The full loss is symmetric: `L = 0.5·D(p1, z2) + 0.5·D(p2, z1)`.

We compute it by hand first. The stop-gradient (added later) only zeros gradients — it does not change the forward *value* — so these forward numbers are just the plain negative cosines. The "collapsed" case (every output identical) gives the floor `-1`, which previews exactly what failure looks like.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# D(p, z) = - cos(p, z)  — the negative cosine similarity (Eqn. 1).
def D(p, z):
    return -F.cosine_similarity(p, z, dim=-1)

# A worked pair of predictions p and targets z.
p1 = torch.tensor([[2.0, 0.0]])
z2 = torch.tensor([[1.0, 1.0]])
p2 = torch.tensor([[0.0, 3.0]])
z1 = torch.tensor([[1.0, 1.0]])

# The symmetric loss L = 0.5*D(p1, z2) + 0.5*D(p2, z1)  (Eqn. 4, forward value).
L = 0.5 * D(p1, z2) + 0.5 * D(p2, z1)
print("worked example:  L =", round(L.item(), 4))               # -0.7071

# The collapsed case: every output is the same vector -> every cosine is 1 -> loss hits the floor -1.
c = torch.tensor([[0.7, 0.7]])
collapsed_loss = 0.5 * D(c, c) + 0.5 * D(c, c)
print("collapsed case:  L =", round(collapsed_loss.item(), 4))   # -1.0

### Step 2 — Define the shared encoder and the asymmetric predictor

SimSiam runs **both** augmented views through the *same* encoder `f` (a small conv backbone plus a projection MLP) to get representations `z`. The asymmetry that makes the method work comes from a small bottleneck **predictor** `h`, applied to only *one* side: `p = h(z)`.

The loss will compare each side's prediction `p` against the *other* side's representation `z`.

In [ ]:
# Shared encoder f: conv backbone + projection MLP -> representation z = f(x).
class Encoder(nn.Module):
    def __init__(self, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 14x14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 7x7
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.proj = nn.Sequential(nn.Linear(32, feat), nn.BatchNorm1d(feat))

    def forward(self, x):
        return self.proj(self.net(x))                                     # z = f(x)


# Predictor h: a small bottleneck MLP on ONE side only -> p = h(z).
class Predictor(nn.Module):
    def __init__(self, feat=64, hid=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat, hid), nn.BatchNorm1d(hid),
            nn.ReLU(), nn.Linear(hid, feat))

    def forward(self, z):
        return self.net(z)                                                # p = h(z)

### Step 3 — Write the SimSiam loss and prepare the data

The whole trick lives in one line: the target side is **detached** (`stopgrad(z) = z.detach()`), so gradients never flow back through it. `stop_grad=True` is the paper's actual method; `stop_grad=False` is the ablation that collapses.

We also set up the two-view augmentation (each image is randomly cropped/blurred twice to make a positive pair) and load a small **unlabelled** MNIST subset to train on.

In [ ]:
# The SimSiam loss (Eqn. 4). stop_grad=True detaches the target (the paper's method);
# stop_grad=False is the ablation that collapses.
def simsiam_loss(z1, z2, p1, p2, stop_grad=True):
    if stop_grad:
        t1, t2 = z1.detach(), z2.detach()      # stopgrad(z) = z.detach()
    else:
        t1, t2 = z1, z2
    return 0.5 * D(p1, t2).mean() + 0.5 * D(p2, t1).mean()              # Eqn. 4

# Two-view augmentation: each image is independently cropped and maybe blurred.
aug = T.Compose([
    T.RandomResizedCrop(28, scale=(0.5, 1.0)),
    T.RandomApply([T.GaussianBlur(3)], 0.5),
    T.ToTensor()])

# An UNLABELLED MNIST subset (torchvision, preinstalled).
raw = torchvision.datasets.MNIST("./data", train=True, download=True)
idx = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]

### Step 4 — Train with vs without the stop-gradient

Now we run the same network twice. After each epoch we record two numbers: the training loss and the **output std** — the spread of the L2-normalised representations. A healthy representation spreads images out (std well above 0); a collapsed one maps every image to nearly the same vector (std → 0).

Watch the contrast: *with* the stop-gradient the loss settles above `-1` and the std stays healthy; *without* it the loss dives to `≈ -1` and the std collapses — the trivial solution.

In [ ]:
# One training run; returns per-epoch (loss, output-std).
def train(stop_grad, epochs=15, B=128):
    torch.manual_seed(0)
    np.random.seed(0)
    enc, pred = Encoder().to(device), Predictor().to(device)
    opt = torch.optim.Adam(list(enc.parameters()) + list(pred.parameters()), lr=1e-3)
    hist = []
    for ep in range(epochs):
        enc.train()
        pred.train()
        perm = np.random.permutation(len(imgs))
        tot = 0.0
        nb = 0
        for s in range(0, len(imgs), B):
            bi = perm[s:s + B]
            x1 = torch.stack([aug(imgs[i]) for i in bi]).to(device)   # view 1
            x2 = torch.stack([aug(imgs[i]) for i in bi]).to(device)   # view 2
            z1, z2 = enc(x1), enc(x2)
            p1, p2 = pred(z1), pred(z2)
            loss = simsiam_loss(z1, z2, p1, p2, stop_grad=stop_grad)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item()
            nb += 1
        # output std: spread of the L2-normalized representation (collapse -> ~0).
        enc.eval()
        with torch.no_grad():
            zz = enc(torch.stack([T.ToTensor()(im) for im in imgs[:512]]).to(device))
            std = F.normalize(zz, dim=1).std(dim=0).mean().item()
        hist.append((tot / nb, std))
    return hist

print("\n=== WITH stop-gradient (the SimSiam method) ===")
for ep, (l, s) in enumerate(train(stop_grad=True)):
    if ep % 3 == 0:
        print(f"  ep {ep:2d}  loss {l:+.3f}  out-std {s:.4f}")

print("=== WITHOUT stop-gradient (ablation -> collapse) ===")
for ep, (l, s) in enumerate(train(stop_grad=False)):
    if ep % 3 == 0:
        print(f"  ep {ep:2d}  loss {l:+.3f}  out-std {s:.4f}")
# With stop-grad: loss settles ABOVE -1 and out-std stays healthy.
# Without stop-grad: loss dives to ~ -1.0 and out-std collapses toward 0  (the trivial solution).
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does the stop-gradient prevent collapse? Compare training loss and output std with vs without it._

### Step 1 — Re-declare the model and data for a self-contained run

This panel re-imports and re-defines the encoder, predictor, loss, augmentation, and data so it can run on its own (independent of the cells above). The definitions match Steps 1–3 exactly; only the variable names are kept compact here.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

# negative cosine (Eqn. 1)
def D(p, z):
    return -F.cosine_similarity(p, z, dim=-1)

class Encoder(nn.Module):
    def __init__(s, feat=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.AdaptiveAvgPool2d(1), nn.Flatten())
        s.proj = nn.Sequential(nn.Linear(32, feat), nn.BatchNorm1d(feat))
    def forward(s, x):
        return s.proj(s.net(x))

class Predictor(nn.Module):
    def __init__(s, feat=64, hid=32):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(feat, hid), nn.BatchNorm1d(hid), nn.ReLU(), nn.Linear(hid, feat))
    def forward(s, z):
        return s.net(z)

def loss_fn(z1, z2, p1, p2, sg):
    if sg:
        t1, t2 = z1.detach(), z2.detach()   # stopgrad(z) = z.detach()
    else:
        t1, t2 = z1, z2
    return 0.5 * D(p1, t2).mean() + 0.5 * D(p2, t1).mean()          # Eqn. 4

aug = T.Compose([T.RandomResizedCrop(28, scale=(0.5, 1.0)),
                 T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
raw = torchvision.datasets.MNIST("./data", train=True, download=True)
idx = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]

### Step 2 — Train both variants and log loss + std per epoch

We run the same network twice — with and without the stop-gradient — and collect the per-epoch `(loss, output-std)` history for each. This is the data behind the comparison: with the stop-gradient the loss stays above `-1` and the std stays healthy; without it, loss → `-1` and std → `0` (collapse).

In [ ]:
def run(sg, epochs=15, B=128):
    torch.manual_seed(0)
    np.random.seed(0)
    enc, pred = Encoder(), Predictor()
    opt = torch.optim.Adam(list(enc.parameters()) + list(pred.parameters()), lr=1e-3)
    hist = []
    for ep in range(epochs):
        enc.train()
        pred.train()
        perm = np.random.permutation(len(imgs))
        tot = 0.0
        nb = 0
        for s0 in range(0, len(imgs), B):
            bi = perm[s0:s0 + B]
            x1 = torch.stack([aug(imgs[i]) for i in bi])
            x2 = torch.stack([aug(imgs[i]) for i in bi])
            z1, z2 = enc(x1), enc(x2)
            p1, p2 = pred(z1), pred(z2)
            l = loss_fn(z1, z2, p1, p2, sg)
            opt.zero_grad()
            l.backward()
            opt.step()
            tot += l.item()
            nb += 1
        enc.eval()
        with torch.no_grad():
            zz = enc(torch.stack([T.ToTensor()(im) for im in imgs[:512]]))
            std = F.normalize(zz, dim=1).std(dim=0).mean().item()
        hist.append((round(tot / nb, 3), round(std, 4)))
    return hist

print("with stop-grad:", run(True))
print("without stop-grad:", run(False))
# with: loss settles above -1, std healthy.  without: loss -> -1, std -> 0 (collapse).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline ablation. You train SimSiam with the stop-gradient and the loss settles around
            $-0.6$ with a healthy output std; then you remove the stop-gradient and the loss drops almost
            instantly to $-1.0$ while the output std falls to near $0$. What has happened, and what does this
            prove about which ingredient prevents collapse?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that loss $=-1$ is the global minimum of a negative-cosine loss, reached when all outputs point the same way. — _$\mathcal{D} = -\cos \ge -1$, with equality iff the vectors are identical in direction; a constant output makes every pair identical._
- Check the output std: near $0$ means every image maps to (nearly) the same vector — the collapsed / trivial solution. — _A useful representation must spread images out (std $\approx 1/\sqrt d$); std $\to 0$ means it carries no information._
- Note the only change was deleting the stop-gradient — the architecture, data, and loss form are identical. — _Holding everything else fixed isolates the stop-gradient as the cause of the difference._

**Answer:** Removing the stop-gradient made both sides chase each other freely, and the easiest way to
                 "match" is to output the same constant vector for every image: every cosine is $1$, so the
                 loss hits its floor $-1$ and the output std collapses to $\approx 0$. Because that was the
                 only change, it proves the stop-gradient — not negatives, not a momentum encoder, not
                 a big batch — is what prevents collapse. Our CODEVIZ panel shows exactly this: stop-grad loss
                 plateaus above $-1$ with healthy std; no-stop-grad loss dives to $-1$ with std $\to 0$.

</details>

**Problem 2.** Your worked example gave $\mathcal{L} = -0.7071$ for $p_1=[2,0],\ z_2=[1,1],\ p_2=[0,3],\ z_1=[1,1]$.
            Now suppose the predictor improves so that $p_1 = [1,1]$ (it now points exactly at $z_2$), with the
            second term unchanged. What is the new loss, and is that closer to or further from collapse?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recompute the first term: $p_1=[1,1]$, $z_2=[1,1]$ are identical directions, so $\cos = 1$ and $\mathcal{D}(p_1,z_2) = -1$. — _Two equal unit-direction vectors have cosine $1$; negating gives $-1$._
- Average with the unchanged second term $\mathcal{D}(p_2,z_1) = -0.7071$: $\mathcal{L} = \tfrac12(-1) + \tfrac12(-0.7071) = -0.8536$. — _The loss is the mean of the two negative cosines._
- Compare $-0.8536$ to the floor $-1$. — _A lower (more negative) loss means better alignment on that term — but $-1$ everywhere would be collapse._

**Answer:** The loss drops to $\mathcal{L} = \tfrac12(-1) + \tfrac12(-0.7071) = \mathbf{-0.8536}$. On
                 this term it is "better" alignment — but note that pushing every term to $-1$ for
                 every image is exactly collapse. The stop-gradient is what lets one term reach $-1$ for a
                 genuinely matching pair without the network discovering it can reach $-1$ everywhere by going
                 constant. A low loss is only healthy if the output std stays near $1/\sqrt d$.

</details>

**Problem 3.** A teammate says "SimSiam is just BYOL without the momentum encoder, and SimCLR without the negatives —
            so it must be strictly weaker and prone to collapse." Where is the misunderstanding, and what single
            ingredient does SimSiam rely on instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what each competitor uses to dodge collapse: SimCLR uses negative pairs (push other images apart); BYOL uses a momentum (slowly-updated) target encoder. — _Both are explicit anti-collapse mechanisms with extra machinery._
- Identify SimSiam's replacement: the stop-gradient turns the same encoder into a fixed target, and the predictor breaks symmetry. — _§4.1 shows stop-grad alone prevents collapse; §4.2 shows the predictor is also required._
- Conclude it is not "weaker," it is minimal: the same anti-collapse effect with neither negatives nor a momentum encoder. — _The abstract reports it learns meaningful representations using none of those three crutches._

**Answer:** The misunderstanding is treating negatives and the momentum encoder as necessary.
                 SimSiam's contribution is showing they are not: a plain shared encoder plus an asymmetric
                 predictor and a stop-gradient on the target avoids collapse on their own. It is the
                 minimal member of the family, not a crippled one — the stop-gradient supplies the anti-collapse
                 pressure that negatives (SimCLR) or a momentum encoder (BYOL) supply in the others.

</details>